# ASAP8 somatic ephys characterization

Extract optical spikes, build ROI/event/synchrony feature tables, and summarize depth-dependent and longitudinal physiology.

Core tables: `sessions`, `rois`, `spikes`, `roi_features`, `spike_features`, `event_features`, and `synchrony_pairs`.


In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
import warnings

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from vip_slap2_analysis.utils.utils import save_figure
from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.voltage.spikes import DETECTOR_VERSION, extract_session_spikes
from vip_slap2_analysis.voltage.analysis import build_analysis_tables, save_analysis_tables

assert DETECTOR_VERSION == "template_v1"
import seaborn as sns
sns.set_style('white')
params = {'legend.fontsize': 'x-large',
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
%matplotlib notebook

## Configuration


In [ ]:
BASE_PATH = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")
SAVE_PATH = Path(r"C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Lab_Meetings\2026-07-28_OPhys_LabMeetingV\figures\voltage_plots")
TARGET_MICE = [852835, 863774]
TARGET_SESSION_LABELS = None
PARADIGMS = ["change_detection_passive"]
EXCLUDE_SESSION_TYPES = ["expression_check", "volume_imaging"]
TRACE_VARIANT = "dff_robust_f0_trial"
REGISTRATION_FILENAME = "roi_identity_registration.csv"
EXCLUDE_INVALID_ROIS = False

SPIKE_KWARGS = dict(
    height_sigma=3.5,
    template_sigma = 4.5,
    prominence_sigma=0.5,
)

BLUE = "#2A7DE2"
PEACH = "#EBA287"
DMD_COLORS = {1: PEACH, 2: BLUE}

## Session registry

Build the session table and resolve paths/metadata used throughout the notebook.


In [ ]:
def _session_label(row):
    if pd.notna(row.get("image_set")) and pd.notna(row.get("image_set_day_index")):
        return f"{row['image_set']}{int(row['image_set_day_index'])}"
    return str(row.get("session_type", row["session_id"]))


def _depth(row, dmd):
    for key in (f"dmd{dmd}_depth", f"dmd{dmd}_depth_um"):
        if key in row and pd.notna(row[key]):
            return float(row[key])
    metadata = row.get("metadata", {}) if isinstance(row.get("metadata", {}), dict) else {}
    return float(metadata.get(f"dmd{dmd}_depth", np.nan))


registry = VIPSessionRegistry.from_basepath(BASE_PATH)
raw = registry.sessions(
    subject_ids=TARGET_MICE,
    paradigms=PARADIGMS,
    exclude_session_types=EXCLUDE_SESSION_TYPES,
).copy()

raw["session_id"] = raw["session_id"].astype(str)
raw["session_datetime"] = pd.to_datetime(
    raw["session_id"].str.extract(r"(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})")[0],
    format="%Y-%m-%d_%H-%M-%S",
    errors="coerce",
)
raw = raw.sort_values(["subject_id", "session_datetime"]).reset_index(drop=True)
raw["session_order"] = raw.groupby("subject_id").cumcount()
raw["session_label"] = raw.apply(_session_label, axis=1)

if TARGET_SESSION_LABELS is not None:
    raw = raw[raw["session_label"].isin(TARGET_SESSION_LABELS)]

rows = []
for _, row in raw.iterrows():
    asset = registry.resolve_assets(row)
    trace_h5 = Path(asset.derived_dir) / "voltage" / f"voltage_session_traces_{TRACE_VARIANT}.h5"
    if not trace_h5.exists():
        warnings.warn(f"{asset.session_id}: missing {trace_h5.name}; skipped")
        continue

    rows.append({
        "subject_id": str(asset.subject_id),
        "session_id": str(asset.session_id),
        "session_label": row["session_label"],
        "session_order": int(row["session_order"]),
        "session_type": row.get("session_type", ""),
        "dmd1_depth_um": _depth(row, 1),
        "dmd2_depth_um": _depth(row, 2),
        "trace_h5": trace_h5,
        "derived_dir": Path(asset.derived_dir),
        "subject_dir": Path(asset.session_dir).parent,
    })

sessions = pd.DataFrame(rows)
if sessions.empty:
    raise RuntimeError("No sessions with processed full-session voltage traces were found.")

display(sessions)


### ROI registry

Create one row per session-specific ROI and retain longitudinal cell identities when available.


In [ ]:
roi_rows = []

for session in sessions.itertuples(index=False):
    with h5py.File(session.trace_h5, "r") as h5:
        for dmd_key in sorted(k for k in h5 if k.startswith("DMD")):
            dmd = int(dmd_key.replace("DMD", ""))
            group = h5[dmd_key]
            n_time = len(group["timebase_sec"])
            dff = group["dff"]

            if dff.shape[-1] == n_time:
                n_rois = dff.shape[0]
            elif dff.shape[0] == n_time:
                n_rois = dff.shape[1]
            else:
                raise ValueError(
                    f"{session.session_id} {dmd_key}: dff shape {dff.shape} does not match timebase"
                )

            valid = (
                group["valid_rois_mask"][:]
                if "valid_rois_mask" in group
                else np.ones(n_rois, bool)
            )
            if len(valid) != n_rois:
                warnings.warn(
                    f"{session.session_id} {dmd_key}: invalid valid_rois_mask length; using all ROIs"
                )
                valid = np.ones(n_rois, bool)

            depth = getattr(session, f"dmd{dmd}_depth_um")
            for roi in range(n_rois):
                roi_rows.append({
                    "subject_id": session.subject_id,
                    "session_id": session.session_id,
                    "session_label": session.session_label,
                    "session_order": session.session_order,
                    "session_type": session.session_type,
                    "dmd": dmd,
                    "roi": roi,
                    "depth_um": depth,
                    "valid_roi": bool(valid[roi]),
                })

rois = pd.DataFrame(roi_rows)

registrations = []
for session in sessions.drop_duplicates("subject_id").itertuples(index=False):
    path = Path(session.subject_dir) / REGISTRATION_FILENAME
    if not path.exists():
        warnings.warn(f"No manual ROI registry for mouse {session.subject_id}: {path}")
        continue

    table = pd.read_csv(
        path,
        dtype={"subject_id": str, "session_id": str, "global_cell_id": str},
    )
    table["subject_id"] = str(session.subject_id)
    keep = [
        c for c in
        ["subject_id", "session_id", "dmd", "roi", "global_cell_id",
         "excluded", "confidence", "notes"]
        if c in table
    ]
    registrations.append(
        table[keep].drop_duplicates(
            ["subject_id", "session_id", "dmd", "roi"],
            keep="last",
        )
    )

if registrations:
    rois = rois.merge(
        pd.concat(registrations, ignore_index=True),
        on=["subject_id", "session_id", "dmd", "roi"],
        how="left",
    )

if "global_cell_id" not in rois:
    rois["global_cell_id"] = ""
if "excluded" not in rois:
    rois["excluded"] = False

rois["global_cell_id"] = rois["global_cell_id"].fillna("").replace("nan", "")
rois["excluded"] = rois["excluded"].fillna(False).map(
    lambda x: x if isinstance(x, (bool, np.bool_))
    else str(x).strip().lower() in {"true", "1", "yes", "y"}
)
rois["manually_registered"] = rois["global_cell_id"].ne("")
rois["cell_id"] = np.where(
    rois["manually_registered"],
    rois["global_cell_id"],
    rois["session_id"] + "_DMD" + rois["dmd"].astype(str) + "_ROI" + rois["roi"].astype(str),
)
rois["included"] = ~rois["excluded"] & (~EXCLUDE_INVALID_ROIS | rois["valid_roi"])
rois = rois.sort_values(["subject_id", "session_order", "dmd", "roi"]).reset_index(drop=True)

display(rois)


## Spike extraction

The detector uses a conservative peak pass followed by template-based rescue of lower-amplitude events with matching waveform shape.


In [ ]:
tables = []

for session in sessions.itertuples(index=False):
    print(f"Processing {session.session_id}, {session.session_label}")
    session_rois = rois[(rois["session_id"] == session.session_id) & rois["included"]]
    roi_lookup = {
        int(dmd): group["roi"].astype(int).tolist()
        for dmd, group in session_rois.groupby("dmd")
    }

    table = extract_session_spikes(
        session.trace_h5,
        rois=roi_lookup,
        **SPIKE_KWARGS,
    )
    table.insert(0, "session_id", session.session_id)
    tables.append(table)

spikes = pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()
spikes = spikes.merge(
    rois[[
        "subject_id", "session_id", "session_label", "session_order", "session_type",
        "dmd", "roi", "depth_um", "cell_id", "global_cell_id",
        "manually_registered",
    ]],
    on=["session_id", "dmd", "roi"],
    how="left",
)

print(f"{len(spikes):,} spikes")
display(spikes.head())


### Spike detection QC

Spot-check detected events and isolated-spike waveforms before downstream feature analysis.


In [ ]:
SESSION_ID = sessions.iloc[4]["session_id"]
DMD = 2
ROIS = [0, 1, 2, 3]
WINDOW = (0, 100)

trace_h5 = sessions.set_index("session_id").loc[SESSION_ID, "trace_h5"]

with h5py.File(trace_h5, "r") as h5:
    g = h5[f"DMD{DMD}"]
    t = g["timebase_sec"][:]
    dff = g["dff"]
    i0, i1 = np.searchsorted(t, WINDOW)
    tw = t[i0:i1]
    traces = np.array([
        dff[roi, i0:i1] if dff.shape[-1] == len(t) else dff[i0:i1, roi]
        for roi in ROIS
    ])

medians = np.median(traces, axis=1)
traces -= medians[:, None]
scale = np.nanpercentile(np.abs(traces), 99)
offsets = np.arange(len(ROIS))[::-1] * 3 * scale

fig, ax = plt.subplots(figsize=(12, 1.5 + len(ROIS)))
for roi, trace, median, offset in zip(ROIS, traces, medians, offsets):
    ax.plot(tw, trace + offset, lw=0.7)
    s = spikes.query(
        "session_id == @SESSION_ID and dmd == @DMD and roi == @roi "
        "and @WINDOW[0] <= spike_time_sec <= @WINDOW[1]"
    )
    ax.scatter(
        s["spike_time_sec"],
        s["peak_dff"] - median + offset,
        marker="v",
        s=18,
        c="k",
        zorder=3,
    )

ax.set(
    yticks=offsets,
    yticklabels=[f"ROI {roi}" for roi in ROIS],
    xlabel="Time (s)",
    title=f"{SESSION_ID} · DMD{DMD}",
)
fig.tight_layout()


In [ ]:
# Spike waveform QC
for roi in ROIS:
    s = spikes.query("session_id == @SESSION_ID and dmd == @DMD and roi == @roi")

    with h5py.File(trace_h5, "r") as h5:
        g = h5[f"DMD{DMD}"]
        t = g["timebase_sec"][:]
        dff = g["dff"]
        trace = dff[roi] if dff.shape[-1] == len(t) else dff[:, roi]
        fs = 1 / np.median(np.diff(t))

    idx = np.searchsorted(t, s["spike_time_sec"])
    search = int(.005 * fs)
    idx = np.array([
        i - search + np.argmax(trace[i-search:i+search+1])
        for i in idx
        if search <= i < len(trace) - search
    ])

    pre, post = int(.005 * fs), int(.010 * fs)
    waves = np.array([
        trace[i-pre:i+post] - np.median(trace[i-pre:i])
        for i in idx
        if pre <= i < len(trace) - post
    ])
    tw = np.arange(-pre, post) / fs * 1000

    plt.figure(figsize=(5, 3))
    plt.plot(tw, waves.T, alpha=.08)
    plt.plot(tw, np.median(waves, axis=0), c="k", lw=2)
    plt.axvline(0, c="k", ls=":")
    plt.xlabel("Time from peak (ms)")
    plt.ylabel("dF/F")
    plt.title(f'ROI{roi} median spike waveform')
    fig.tight_layout()


In [ ]:
sessions.iloc[4]

## Feature tables

Build per-ROI, per-spike, per-event, and within-session pairwise synchrony summaries. Waveform summaries use isolated spikes; synchrony is computed only within a session.


In [ ]:
FEATURE_KWARGS = dict(
    isolation_ms=50.0,
    waveform_pre_ms=5.0,
    waveform_post_ms=15.0,
    waveform_peak_refine_ms=2.0,
    waveform_baseline_window_ms=(-5.0, -1.0),
    waveform_auc_window_ms=(-2.0, 10.0),
    compound_isi_ms=50.0,
    burst_min_spikes=3,
    spike_sttc_dt_ms=40.0,
    burst_sttc_dt_ms=40.0,
    spike_count_bin_ms=100.0,
)

SAVE_ANALYSIS_TABLES = True
ANALYSIS_SUBDIR = Path("voltage") / "ephys_characterization"


In [ ]:
analysis_tables = build_analysis_tables(
    sessions,
    rois,
    spikes,
    **FEATURE_KWARGS,
)

roi_features = analysis_tables["roi_features"]
spike_features = analysis_tables["spike_features"]
event_features = analysis_tables["event_features"]
synchrony_pairs = analysis_tables["synchrony_pairs"]

if SAVE_ANALYSIS_TABLES:
    save_analysis_tables(
        sessions,
        analysis_tables,
        subdir=ANALYSIS_SUBDIR,
        parameters=FEATURE_KWARGS,
    )

for name, table in analysis_tables.items():
    print(f"{name}: {len(table):,} rows")

display(roi_features.head())
display(synchrony_pairs.head())


### Example spike trains and events

Inspect representative spike trains, isolated-spike waveforms, and compound events.


In [ ]:
# Example spike train + isolated spike waveform
subject_id = '852835'
session_label = 'B1'
SESSION_ID = sessions[(sessions['subject_id']==subject_id)&(sessions['session_label']==session_label)]['session_id'].values[0]
DMD = 1
ROI = 2
EVENT_INDEX = 10         # choose event to show on left

EVENT_WINDOW_MS = (-20, 120)
WAVEFORM_WINDOW_MS = (-10, 15)

COLOR = DMD_COLORS[DMD]
# ---------- Load trace ----------
trace_h5 = sessions.set_index("session_id").loc[SESSION_ID, "trace_h5"]

with h5py.File(trace_h5, "r") as h5:
    g = h5[f"DMD{DMD}"]
    t = g["timebase_sec"][:]
    dff = g["dff"]
    trace = dff[ROI] if dff.shape[-1] == len(t) else dff[:, ROI]
    fs = 1 / np.median(np.diff(t))


# ---------- ROI spikes ----------
s = spike_features.query(
    "session_id == @SESSION_ID and dmd == @DMD and roi == @ROI"
).copy()

event = s.query("event_index == @EVENT_INDEX")
t0 = event["spike_time_sec"].min()


# ---------- Left: example event ----------
window = t0 + np.array(EVENT_WINDOW_MS) / 1000
mask = (t >= window[0]) & (t <= window[1])

event_spikes = s[
    (s["spike_time_sec"] >= window[0]) &
    (s["spike_time_sec"] <= window[1])
]


# ---------- Right: isolated spike waveforms ----------
isolated = s[s["is_isolated"]]

pre = int(abs(WAVEFORM_WINDOW_MS[0]) / 1000 * fs)
post = int(WAVEFORM_WINDOW_MS[1] / 1000 * fs)

idx = np.searchsorted(t, isolated["spike_time_sec"])

waves = np.array([
    trace[i-pre:i+post] - np.median(trace[i-pre:i])
    for i in idx
    if pre <= i < len(trace) - post
])

tw = np.arange(-pre, post) / fs * 1000


# ---------- Plot ----------
fig, ax = plt.subplots(
    1, 2,
    figsize=(6.5, 3),
    gridspec_kw={"width_ratios": [1.5, 1]},sharey=False
)
for axis in ax.flatten():
    axis.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
    axis.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
    for spine in ['left','bottom','top','right']:
        axis.spines[spine].set_linewidth(2)
# example compound/single event
ax[0].plot((t[mask] - t0) * 1000, trace[mask], lw=2,color=COLOR)

ax[0].scatter(
    (event_spikes["spike_time_sec"] - t0) * 1000,
    np.interp(event_spikes["spike_time_sec"], t, trace),
    marker="v",
    c="fuchsia",
    s=50,
    zorder=3,
)

ax[0].axvline(0, c="k", ls="--", lw=1)
ax[0].set_xlabel('Time (ms) from first spike',fontsize=15)
ax[0].set_ylabel('\u0394F/F$_{0}$',fontsize=15)
ax[0].set_title('Single spike example',fontsize=15)
# isolated spike morphology
ax[1].plot(tw, waves.T, alpha=.06,color=COLOR)
ax[1].plot(tw, np.median(waves, axis=0), c="k", lw=2)
ax[1].axvline(0, c="k", ls="--", lw=1)
ax[1].set_xlabel('Time (ms) from spike',fontsize=15)
ax[1].set_title('Median spike',fontsize=15)

fig.tight_layout()
filen = f'{SESSION_ID}_DMD{DMD}_ROI{ROI}_spikewaveform'
save_figure(fig,os.path.join(SAVE_PATH,filen),formats=['.pdf'],dpi=300)

In [ ]:
# Example event + isolated spike waveform

subject_id = "852835"
session_label = "B1"

SESSION_ID = sessions[
    (sessions["subject_id"] == subject_id) &
    (sessions["session_label"] == session_label)
]["session_id"].values[0]

DMD = 2
ROI = 0

EVENT_TYPE = "burst"       # "singleton", "doublet", or "burst"
EVENT_NUMBER = 501           # nth event of this type

EVENT_WINDOW_MS = (-20, 120)
WAVEFORM_WINDOW_MS = (-10, 15)

COLOR = DMD_COLORS[DMD]


# ---------- Select example event ----------

candidates = (
    event_features.query(
        "session_id == @SESSION_ID and dmd == @DMD and roi == @ROI "
        "and event_type == @EVENT_TYPE"
    )
    .sort_values("event_start_sec")
    .reset_index(drop=True)
)

display(
    candidates[
        ["event_index", "event_start_sec", "n_spikes", "event_duration_ms"]
    ].sort_values(by='event_duration_ms')
)

event_info = candidates.iloc[EVENT_NUMBER]
EVENT_INDEX = event_info["event_index"]
t0 = event_info["event_start_sec"]


# ---------- Load trace ----------

trace_h5 = sessions.set_index("session_id").loc[SESSION_ID, "trace_h5"]

with h5py.File(trace_h5, "r") as h5:
    g = h5[f"DMD{DMD}"]
    t = g["timebase_sec"][:]
    dff = g["dff"]
    trace = dff[ROI] if dff.shape[-1] == len(t) else dff[:, ROI]
    fs = 1 / np.median(np.diff(t))


# ---------- ROI spikes ----------

s = spike_features.query(
    "session_id == @SESSION_ID and dmd == @DMD and roi == @ROI"
).copy()

# Only spikes belonging to the selected event
event_spikes = s.query("event_index == @EVENT_INDEX")

window = t0 + np.array(EVENT_WINDOW_MS) / 1000
mask = (t >= window[0]) & (t <= window[1])


# ---------- Isolated spike waveforms ----------

isolated = s[s["is_isolated"]]

pre = int(abs(WAVEFORM_WINDOW_MS[0]) / 1000 * fs)
post = int(WAVEFORM_WINDOW_MS[1] / 1000 * fs)

idx = np.searchsorted(t, isolated["spike_time_sec"])

waves = np.array([
    trace[i-pre:i+post] - np.median(trace[i-pre:i])
    for i in idx
    if pre <= i < len(trace) - post
])

tw = np.arange(-pre, post) / fs * 1000


# ---------- Plot ----------

fig, ax = plt.subplots(
    1, 2,
    figsize=(6.5, 3),
    gridspec_kw={"width_ratios": [1.5, 1]},
)

for axis in ax:
    axis.tick_params(axis="x", top=False, labelsize=12)
    axis.tick_params(axis="y", right=False, labelsize=12)
    for spine in axis.spines.values():
        spine.set_linewidth(2)


# Example event
ax[0].plot(
    (t[mask] - t0) * 1000,
    trace[mask],
    lw=2,
    color=COLOR,
)

ax[0].scatter(
    (event_spikes["spike_time_sec"] - t0) * 1000,
    np.interp(event_spikes["spike_time_sec"], t, trace),
    marker="v",
    c="fuchsia",
    s=50,
    zorder=3,
)

ax[0].axvline(0, c="k", ls="--", lw=1)
ax[0].set_xlabel("Time (ms) from first spike", fontsize=15)
ax[0].set_ylabel(r"$\Delta$F/F$_0$", fontsize=15)
ax[0].set_title(
    f"Single spike example",
    fontsize=15,
)


# Isolated spike morphology
ax[1].plot(tw, waves.T, alpha=.06, color=COLOR)
ax[1].plot(tw, np.median(waves, axis=0), c="k", lw=2)
ax[1].axvline(0, c="k", ls="--", lw=1)

ax[1].set_xlabel("Time (ms) from spike", fontsize=15)
ax[1].set_title("Median isolated spike", fontsize=15)

fig.tight_layout()

filen = f"{SESSION_ID}_DMD{DMD}_ROI{ROI}_{EVENT_TYPE}_spikewaveform"
save_figure(
    fig,
    os.path.join(SAVE_PATH, filen),
    formats=[".pdf"],
    dpi=300,
)

## Population analyses

Summarize firing, bursting, waveform shape, depth dependence, and longitudinal changes.


### Spike rate and bursting

Compare firing and burst statistics across neurons and depth groups.


In [ ]:
METRICS = [
    "spike_rate_hz",
    "burst_event_rate_hz",
    "median_spike_half_width_ms",
    "median_spike_rise10_90_ms",
    "median_spike_decay90_10_ms",
    "median_spike_amplitude_dff",
    "median_spike_auc_dff_ms",
    "compound_event_fraction",
]

METRIC_LABELS = {
    "spike_rate_hz": "Spike rate (Hz)",
    "burst_event_rate_hz": "Burst rate (Hz)",
    "median_spike_half_width_ms": "Half-width (ms)",
    "median_spike_rise10_90_ms": "10–90% rise (ms)",
    "median_spike_decay90_10_ms": "Decay (ms)",
    "median_spike_amplitude_dff": "Amplitude (ΔF/F)",
    "median_spike_auc_dff_ms": "AUC (ΔF/F·ms)",
    "compound_event_fraction": "Compound event fraction",
}

DEPTH_COLORS = {
    "superficial": "#EBA287",
    "middle": "#d1e2b0",
    "deep": "#7bbcd5",
}

df = roi_features.dropna(subset=METRICS + ["depth_um"]).copy()

df["depth_group"] = np.select(
    [df["depth_um"] < 100, df["depth_um"] > 150],
    ["superficial", "deep"],
    default="middle",
)

df["color"] = df["depth_group"].map(DEPTH_COLORS)

n = len(METRICS)
fig, axes = plt.subplots(
    n, n,
    figsize=(2.0 * n, 2.0 * n),
)

for i, ymetric in enumerate(METRICS):
    for j, xmetric in enumerate(METRICS):
        ax = axes[i, j]

        if i == j:
            for group in ["superficial", "middle", "deep"]:
                vals = df.loc[df["depth_group"] == group, xmetric]
                ax.hist(
                    vals,
                    bins=15,
                    histtype="step",
                    lw=1.5,
                    color=DEPTH_COLORS[group],
                )

        elif i > j:
            ax.scatter(
                df[xmetric],
                df[ymetric],
                c=df["color"],
                s=18,
                alpha=0.9,
                linewidth=0,
            )

        else:
            ax.axis("off")
            continue

        ax.tick_params(axis="both", labelsize=7)

        if i == n - 1:
            ax.set_xlabel(METRIC_LABELS[xmetric], fontsize=8)
        else:
            ax.set_xticklabels([])

        if j == 0:
            ax.set_ylabel(METRIC_LABELS[ymetric], fontsize=8)
        else:
            ax.set_yticklabels([])

        for spine in ax.spines.values():
            spine.set_linewidth(1.25)

handles = [
    plt.Line2D([], [], marker="o", ls="", color="#EBA287", label="<100 µm"),
    plt.Line2D([], [], marker="o", ls="", color="#d1e2b0", label="100–150 µm"),
    plt.Line2D([], [], marker="o", ls="", color="#7bbcd5", label=">150 µm"),
]

fig.legend(
    handles=handles,
    title="Depth",
    frameon=False,
    loc="upper right",
)

fig.tight_layout()

In [ ]:
# Spike rate vs burst rate across neurons

df = roi_features.dropna(
    subset=["spike_rate_hz", "burst_event_rate_hz", "depth_um"]
).copy()

df["color"] = np.select([df["depth_um"] < 100,df["depth_um"] > 150,],["#EBA287","#7bbcd5"],default="#d1e2b0")

fig, ax = plt.subplots(figsize=(3, 3))
sns.despine()

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

ax.scatter(
    df["spike_rate_hz"],
    df["burst_event_rate_hz"],
    c=df["color"],
    s=30,
    alpha=1.0,
    linewidth=0.5,
)

ax.set_xlabel(" Avg. spike rate (Hz)", fontsize=15)
ax.set_ylabel("Avg. burst rate (Hz)", fontsize=15)

ax.tick_params(axis="both", labelsize=12)

for spine in ax.spines.values():
    spine.set_linewidth(2)
    
for label, color in [("<100 µm", "#EBA287"),("100–150 µm", "#d1e2b0"),(">150 µm", "#7bbcd5"),]:
    ax.scatter([], [], color=color, s=30, label=label)
    
ax.legend(title="Depth", frameon=False,fontsize=8,handletextpad=0.1,loc='upper left')
fig.tight_layout()
filen = 'SpikeRate_BurstRate'
save_figure(fig,os.path.join(SAVE_PATH,filen),formats=['.pdf'],dpi=300)

In [ ]:
# Spike rate vs burst rate with marginal depth-group histograms

x = "spike_rate_hz"
y = "burst_event_rate_hz"

df = roi_features.dropna(
    subset=[x,y, "depth_um"]
).copy()

df["depth_group"] = np.select(
    [df["depth_um"] < 100, df["depth_um"] > 150],
    ["<100 µm", ">150 µm"],
    default="100–150 µm",
)

DEPTH_COLORS = {
    "<100 µm": "#EBA287",
    "100–150 µm": "#d1e2b0",
    ">150 µm": "#7bbcd5",
}

df["color"] = df["depth_group"].map(DEPTH_COLORS)


# ---------- Figure ----------
fig = plt.figure(figsize=(4, 4))

gs = fig.add_gridspec(
    2, 2,
    width_ratios=[4, 1],
    height_ratios=[1, 4],
    hspace=0.05,
    wspace=0.05,
)

ax_top = fig.add_subplot(gs[0, 0])
ax = fig.add_subplot(gs[1, 0])
ax_right = fig.add_subplot(gs[1, 1])

# Explicitly leave room for large axis labels
fig.subplots_adjust(
    left=0.22,
    bottom=0.20,
    right=0.96,
    top=0.96,
)

sns.despine(ax=ax)
sns.despine(ax=ax_top, bottom=True)
sns.despine(ax=ax_right, left=True)

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
# ---------- Main scatter ----------
ax.scatter(
    df[x],
    df[y],
    c=df["color"],
    s=30,
    alpha=1.0,
    linewidth=0.5,
)

ax.set_xlabel("Avg. spike rate (Hz)", fontsize=15)
ax.set_ylabel("Avg. burst rate (Hz)", fontsize=15)

# ax.set_ylabel("Avg. spike amp. (\u0394F/$F_{0}$)", fontsize=15)
# ax.set_xlabel("Spike width (ms)", fontsize=15)

ax.tick_params(axis="both", labelsize=12)

for spine in ax.spines.values():
    spine.set_linewidth(2)


# ---------- Marginal KDEs ----------
for group, color in DEPTH_COLORS.items():
    d = df[df["depth_group"] == group]

    sns.kdeplot(
        data=d,
        x=x,
        ax=ax_top,
        color=color,
        lw=2,
        fill=True,
        common_norm=False,
#         clip=ax.get_xlim(),
    )

    sns.kdeplot(
        data=d,
        y=y,
        ax=ax_right,
        color=color,
        lw=2,
        fill=True,
        common_norm=False,
#         clip=ax.get_ylim(),
    )


# Match marginal axes to scatter
ax_top.set_xlim(ax.get_xlim())
ax_right.set_ylim(ax.get_ylim())


# ---------- Clean marginal axes ----------
ax_top.set(
    xlabel=None,
    ylabel=None,
    xticks=[],
    yticks=[],
)

ax_right.set(
    xlabel=None,
    ylabel=None,
    xticks=[],
    yticks=[],
)

for spine in ax_top.spines.values():
    spine.set_visible(False)

for spine in ax_right.spines.values():
    spine.set_visible(False)

# ---------- Legend ----------
for label, color in DEPTH_COLORS.items():
    ax.scatter([], [], color=color, s=30, label=label)

ax.legend(
    title="Depth",
    frameon=False,
    fontsize=8,
    handletextpad=0.1,
    loc="upper left",
)
fig.tight_layout()

filen = "SpikeRate_BurstRate_marginals"
# filen = "AMP_FWHM_marginals"
save_figure(
    fig,
    os.path.join(SAVE_PATH, filen),
    formats=[".pdf"],
    dpi=300,
)

### Depth-dependent spike features

Compare session-by-ROI feature distributions across the three soma-depth bins.


In [ ]:
# Compound event fraction by depth
# Each session × ROI is treated as an independent observation

from scipy.stats import kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests

GROUP_ORDER = ["<100 µm", "100–150 µm", ">150 µm"]
DEPTH_COLORS = {"<100 µm": "#EBA287","100–150 µm": "#d1e2b0",">150 µm": "#7bbcd5"}

y = "median_spike_half_width_ms"

# ---------- Data ----------
plot_df = roi_features.dropna(subset=[y, "depth_um"]).copy()
plot_df["depth_group"] = np.select([plot_df["depth_um"] < 100,plot_df["depth_um"] > 150,],["<100 µm",">150 µm",],default="100–150 µm",)

# ---------- Statistics ----------
groups = {g: plot_df.loc[plot_df["depth_group"] == g, y].values for g in GROUP_ORDER}

H, p_kw = kruskal(*groups.values())

pairs = [("<100 µm", "100–150 µm"),("<100 µm", ">150 µm"),("100–150 µm", ">150 µm"),]
p_raw = [mannwhitneyu(groups[a],groups[b],alternative="two-sided",).pvalue for a, b in pairs]
p_adj = multipletests(p_raw,method="holm",)[1]

# ---------- Plot ----------
fig, ax = plt.subplots(figsize=(3.5, 3.5))
sns.despine()

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

sns.violinplot(data=plot_df,x="depth_group",y=y,order=GROUP_ORDER,
               palette=DEPTH_COLORS,linewidth=2,ax=ax,width=0.5)

import matplotlib.colors as mcolors

# Separate face and edge opacity
for violin, color in zip(
    ax.collections[:3],
    [DEPTH_COLORS[g] for g in GROUP_ORDER],
):
    violin.set_facecolor(mcolors.to_rgba(color, 0.25))
    violin.set_edgecolor("black")
    violin.set_linewidth(1.5)

sns.stripplot(data=plot_df,x="depth_group",y=y,
              order=GROUP_ORDER,palette=DEPTH_COLORS,size=5,alpha=1,jitter=0.18,ax=ax)

ax.set_xlabel("")
# ax.set_ylabel("Burst duration (ms)", fontsize=14)
# ax.set_ylabel("Burst rate (Hz)", fontsize=14)
# ax.set_ylabel("Avg. spike amp. (\u0394F/F$_{0}$)", fontsize=14)
ax.set_ylabel("Avg. spike width (ms)", fontsize=14)
ax.tick_params(axis="both", labelsize=10)

for spine in ax.spines.values():
    spine.set_linewidth(2)

# ---------- Significance annotations ----------
def p_label(p):
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


yrange = plot_df[y].max() - \
         plot_df[y].min()

y0 = plot_df[y].max() + 0.06 * yrange
step = 0.15 * yrange

positions = {group: i for i, group in enumerate(GROUP_ORDER)}

for k, ((a, b), p) in enumerate(zip(pairs, p_adj)):

    x1, x2 = positions[a], positions[b]
    y = y0 + k * step
    h = 0.1 * yrange

    ax.plot([x1, x1, x2, x2],[y, y + h, y + h, y],c="k",lw=1,)
    ax.text((x1 + x2) / 2,y + h,p_label(p),ha="center",va="bottom",fontsize=11,)
    
ax.set_ylim(top=y0 + (len(pairs) + 1) * step)
ax.set_xlabel('Depth bin')
ax.text(0.02,0.98,f"Kruskal–Wallis p = {p_kw:.3g}",transform=ax.transAxes,ha="left",va="top",fontsize=9,)
fig.tight_layout()
filen = 'SpikeWidth_dists'
save_figure(fig,os.path.join(SAVE_PATH, filen),formats=[".pdf"],dpi=300,)

### Longitudinal feature trajectories

Track registered neurons across sessions; line color is fixed by each neuron's median longitudinal depth.


In [ ]:
# Longitudinal feature trajectories across sessions
# One line = one registered neuron

GROUP_ORDER = ["<100 µm", "100–150 µm", ">150 µm"]

DEPTH_COLORS = {
    "<100 µm": "#EBA287",
    "100–150 µm": "#d1e2b0",
    ">150 µm": "#7bbcd5",
}

METRICS = {
    "median_spike_amplitude_dff": {
        "ylabel": "Avg. spike amplitude (ΔF/F)",
        "filename": "SpikeAmplitude_sessions",
    },
    "median_spike_half_width_ms": {
        "ylabel": "Avg. FWHM (ms)",
        "filename": "SpikeFWHM_sessions",
    },
    "burst_event_rate_hz": {
        "ylabel": "Burst rate (Hz)",
        "filename": "BurstRate_sessions",
    },
    "median_burst_duration_ms": {
        "ylabel": "Burst duration (ms)",
        "filename": "BurstDuration_sessions",
    },
    "spike_rate_hz": {
        "ylabel": "Spike rate (Hz)",
        "filename": "SpikeRate_sessions",
    },
}


# ============================================================
# SWITCH METRIC HERE
# ============================================================

y = "spike_rate_hz"

ylabel = METRICS[y]["ylabel"]
filen = METRICS[y]["filename"]


# ---------- Data ----------

plot_df = roi_features.dropna(
    subset=[y, "depth_um", "global_cell_id"]
).copy()

# Exclude G1
plot_df = plot_df[
    plot_df["session_label"].astype(str) != "G1"
].copy()

# Stable longitudinal neuron identifier
plot_df["neuron_id"] = (
    plot_df["subject_id"].astype(str)
    + ":"
    + plot_df["global_cell_id"].astype(str)
)

# Remove blank / invalid registrations
plot_df = plot_df[
    plot_df["global_cell_id"].astype(str).notna()
    & ~plot_df["global_cell_id"].astype(str).isin(["", "nan", "None"])
].copy()


# ---------- Keep neurons recorded on >=2 sessions ----------

n_sessions = (
    plot_df.groupby("neuron_id")["session_id"]
    .nunique()
)

keep_neurons = n_sessions[n_sessions >= 2].index

plot_df = plot_df[
    plot_df["neuron_id"].isin(keep_neurons)
].copy()


# ---------- Assign one depth group per neuron ----------
# Using median longitudinal depth keeps line color constant across days.

neuron_depth = (
    plot_df.groupby("neuron_id")["depth_um"]
    .median()
)

plot_df["neuron_depth_um"] = plot_df["neuron_id"].map(neuron_depth)

plot_df["depth_group"] = np.select(
    [
        plot_df["neuron_depth_um"] < 100,
        plot_df["neuron_depth_um"] > 150,
    ],
    [
        "<100 µm",
        ">150 µm",
    ],
    default="100–150 µm",
)


# ---------- Determine session order ----------

session_order = (
    plot_df[["session_label", "session_order"]]
    .drop_duplicates()
    .groupby("session_label", as_index=False)["session_order"]
    .median()
    .sort_values("session_order")
)

SESSION_ORDER = session_order["session_label"].tolist()

plot_df["session_label"] = pd.Categorical(
    plot_df["session_label"],
    categories=SESSION_ORDER,
    ordered=True,
)

plot_df = plot_df.sort_values(
    ["neuron_id", "session_order"]
)


# ---------- Plot ----------

fig, ax = plt.subplots(figsize=(3.5, 3.5))

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=10)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=10)

sns.despine()

x_positions = {
    label: i
    for i, label in enumerate(SESSION_ORDER)
}

for neuron_id, cell in plot_df.groupby("neuron_id"):

    cell = cell.sort_values("session_order")

    group = cell["depth_group"].iloc[0]
    color = DEPTH_COLORS[group]

    x = cell["session_label"].map(x_positions).astype(float).to_numpy()
    values = cell[y].to_numpy()

    # trajectory
    ax.plot(
        x,
        values,
        color=color,
        alpha=0.35,
        lw=1,
        zorder=1,
    )

    # observations
    ax.scatter(
        x,
        values,
        s=28,
        color=color,
        edgecolor="none",
        alpha=0.9,
        zorder=2,
    )


# ---------- Group median trajectory ----------

for group in GROUP_ORDER:

    g = plot_df[
        plot_df["depth_group"] == group
    ]

    med = (
        g.groupby("session_label", observed=True)[y]
        .median()
        .reindex(SESSION_ORDER)
    )

    x = np.arange(len(SESSION_ORDER))
    valid = med.notna()

    ax.plot(
        x[valid],
        med[valid],
        color=DEPTH_COLORS[group],
        lw=3,
        marker="o",
        markersize=6,
        markeredgecolor="black",
        markeredgewidth=0.8,
        label=group,
        zorder=4,
    )


# ---------- Aesthetics ----------

ax.set_xticks(np.arange(len(SESSION_ORDER)))
ax.set_xticklabels(SESSION_ORDER)

ax.set_xlabel("Session", fontsize=15)
ax.set_ylabel(ylabel, fontsize=15)

ax.tick_params(
    axis="both",
    labelsize=11,
)

for spine in ax.spines.values():
    spine.set_linewidth(2)

ax.legend(
    frameon=False,
    title="Depth",
    fontsize=9,
    title_fontsize=9,
)

fig.tight_layout()

save_figure(
    fig,
    os.path.join(SAVE_PATH, filen),
    formats=[".pdf"],
    dpi=300,
)

In [ ]:
from itertools import combinations
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

# plot_df is the dataframe used for the longitudinal plot
# y is your currently selected metric

MIN_PAIRS = 4

session_pairs = list(combinations(SESSION_ORDER, 2))

results = []

for depth_group in GROUP_ORDER:

    g = plot_df[
        plot_df["depth_group"] == depth_group
    ].copy()

    # neuron × session matrix
    wide = g.pivot_table(
        index="neuron_id",
        columns="session_label",
        values=y,
        aggfunc="median",
        observed=True,
    )

    for session_a, session_b in session_pairs:

        if session_a not in wide.columns or session_b not in wide.columns:
            continue

        paired = wide[[session_a, session_b]].dropna()

        if len(paired) < MIN_PAIRS:
            continue

        stat, p = wilcoxon(
            paired[session_a],
            paired[session_b],
            alternative="two-sided",
        )

        results.append({
            "depth_group": depth_group,
            "session_a": session_a,
            "session_b": session_b,
            "n_pairs": len(paired),
            "statistic": stat,
            "p_raw": p,
            "median_change": np.median(
                paired[session_b] - paired[session_a]
            ),
        })


stats_df = pd.DataFrame(results)

# Holm correction across every comparison displayed
if len(stats_df):
    stats_df["p_adj"] = multipletests(
        stats_df["p_raw"],
        method="holm",
    )[1]

    stats_df["significance"] = pd.cut(
        stats_df["p_adj"],
        bins=[-np.inf, 0.001, 0.01, 0.05, np.inf],
        labels=["***", "**", "*", "ns"],
    )

stats_df

### Spike waveform shape

Compare isolated-spike waveform shape across depth groups, then inspect one longitudinally tracked example neuron per depth bin.


In [ ]:
# Waveforms by depth group: one panel per depth, all ROI median waveforms overlaid

WAVEFORM_WINDOW_MS = (-10, 20)
ALIGN_WINDOW_MS = 5
MIN_SPIKES = 10
NORMALIZE = False          # True = compare shape only
PLOT_GROUP_MEDIAN = True   # overlay median across ROIs in black

DEPTH_COLORS = {
    "superficial": "#EBA287",   # <100 um
    "middle": "#d1e2b0",        # 100-150 um
    "deep": "#7bbcd5",          # >150 um
}

DEPTH_ORDER = ["superficial", "middle", "deep"]
DEPTH_LABELS = {
    "superficial": "<100 µm",
    "middle": "100–150 µm",
    "deep": ">150 µm",
}

def depth_group(depth):
    if depth < 100:
        return "superficial"
    if depth > 150:
        return "deep"
    return "middle"


# ---------- Precompute lookups ----------
session_to_h5 = sessions.set_index("session_id")["trace_h5"].to_dict()

isolated_spikes = spike_features[spike_features["is_isolated"]].copy()
spike_groups = {
    key: grp["spike_time_sec"].to_numpy()
    for key, grp in isolated_spikes.groupby(["session_id", "dmd", "roi"])
}

cells = roi_features.copy()
cells = cells[np.isfinite(cells["depth_um"])].copy()
cells["depth_group"] = cells["depth_um"].apply(depth_group)

# Only keep ROIs that actually have isolated spikes
cells["key"] = list(zip(cells["session_id"], cells["dmd"], cells["roi"]))
cells = cells[cells["key"].isin(spike_groups)].copy()


# ---------- Collect waveforms ----------
group_waveforms = {k: [] for k in DEPTH_ORDER}
tw_ref = None

for session_id, session_cells in cells.groupby("session_id"):
    trace_h5 = session_to_h5[session_id]

    with h5py.File(trace_h5, "r") as h5:
        for cell in session_cells.itertuples():
            key = (cell.session_id, cell.dmd, cell.roi)
            spike_times = spike_groups.get(key, None)
            if spike_times is None or len(spike_times) < MIN_SPIKES:
                continue

            dmd = int(cell.dmd)
            roi = int(cell.roi)
            depth_grp = cell.depth_group

            g = h5[f"DMD{dmd}"]
            t = g["timebase_sec"][:]
            dff = g["dff"]
            trace = dff[roi] if dff.shape[-1] == len(t) else dff[:, roi]
            fs = 1 / np.median(np.diff(t))

            search = int(round(ALIGN_WINDOW_MS / 1000 * fs))
            pre = int(round(abs(WAVEFORM_WINDOW_MS[0]) / 1000 * fs))
            post = int(round(WAVEFORM_WINDOW_MS[1] / 1000 * fs))

            idx = np.searchsorted(t, spike_times)
            idx = idx[(idx >= search) & (idx < len(trace) - search)]

            if len(idx) < MIN_SPIKES:
                continue

            # Align to local raw dF/F maximum
            aligned_idx = np.array([
                i - search + np.argmax(trace[i-search:i+search+1])
                for i in idx
                if pre <= i < len(trace) - post
            ], dtype=int)

            if len(aligned_idx) < MIN_SPIKES:
                continue

            waves = np.array([
                trace[i-pre:i+post] - np.median(trace[i-pre:i])
                for i in aligned_idx
                if pre <= i < len(trace) - post
            ])

            if len(waves) < MIN_SPIKES:
                continue

            waveform = np.median(waves, axis=0)

            if NORMALIZE:
                peak = np.nanmax(waveform)
                if peak > 0:
                    waveform = waveform / peak
                else:
                    continue

            group_waveforms[depth_grp].append(waveform)

            if tw_ref is None:
                tw_ref = np.arange(-pre, post) / fs * 1000

In [ ]:

# ---------- Plot ----------
fig, axes = plt.subplots(3, 1, figsize=(3, 6), sharex=True, sharey=True)

for ax, grp in zip(axes, DEPTH_ORDER):
    sns.despine(ax=ax,left=True,bottom=True)
    waves = group_waveforms[grp]
    color = DEPTH_COLORS[grp]

    for w in waves:
        ax.plot(tw_ref, w, color=color, lw=1.0, alpha=0.35)

    if PLOT_GROUP_MEDIAN and len(waves):
        ax.plot(tw_ref, np.median(np.vstack(waves), axis=0), color="k", lw=2.5)

    ax.axvline(0, color="k", ls="--", lw=1)
#     ax.set_title(f"{DEPTH_LABELS[grp]}  (n={len(waves)} ROIs)", fontsize=12)
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    for spine in ax.spines.values():
        spine.set_linewidth(2)

axes[0].plot([10,15],[0.1,0.1],color='k',solid_capstyle='round',lw=2)
axes[0].text(10,0.055,'5 ms',fontsize=12)
axes[0].plot([10,10],[0.1,0.2],color='k',solid_capstyle='round',lw=2)
axes[0].text(9.5,0.11,'0.1\n\u0394F/F$_{0}$',fontsize=12,rotation=0,ha='right')

fig.tight_layout()

filen = "Example_mean_waveforms"
save_figure(
    fig,
    os.path.join(SAVE_PATH, filen),
    formats=[".pdf"],
    dpi=300,
)

#### Longitudinal example waveforms

Rows are depth groups and columns are A0–B2. By default the code selects the best-covered registered neuron in each depth bin; edit `EXAMPLE_NEURONS` to curate examples.


In [ ]:
# Longitudinal waveform grid: one registered neuron per depth group
# Individual isolated spikes shown behind each neuron/day median.

SESSION_GRID = ["A0", "A1", "A2", "B0", "B1", "B2"]

EXAMPLE_NEURONS = {
    "superficial": None,
    "middle": None,
    "deep": None,
}

MAX_PLOTTED_SPIKES = 1000
SPIKE_LINEWIDTH = 0.35
SPIKE_ALPHA = 0.018
RNG = np.random.default_rng(8)

# ---------- Candidate longitudinal cells ----------
wave_cells = roi_features[
    roi_features["session_label"].astype(str).isin(SESSION_GRID)
    & np.isfinite(roi_features["depth_um"])
    & roi_features["global_cell_id"].notna()
].copy()

wave_cells = wave_cells[
    ~wave_cells["global_cell_id"].astype(str).isin(["", "nan", "None"])
].copy()

wave_cells["neuron_id"] = (
    wave_cells["subject_id"].astype(str) + ":" +
    wave_cells["global_cell_id"].astype(str)
)

# Keep depth assignment stable across days.
neuron_depth = wave_cells.groupby("neuron_id")["depth_um"].median()
wave_cells["neuron_depth_um"] = wave_cells["neuron_id"].map(neuron_depth)
wave_cells["depth_group"] = wave_cells["neuron_depth_um"].apply(depth_group)

wave_cells["key"] = list(zip(
    wave_cells["session_id"],
    wave_cells["dmd"],
    wave_cells["roi"],
))
wave_cells["n_isolated"] = wave_cells["key"].map(
    lambda k: len(spike_groups.get(k, []))
)
wave_cells = wave_cells[wave_cells["n_isolated"] >= MIN_SPIKES].copy()

wave_cells = (
    wave_cells
    .sort_values("n_isolated", ascending=False)
    .drop_duplicates(["neuron_id", "session_label"])
)

# ---------- Select one well-covered neuron per depth group ----------
coverage = (
    wave_cells
    .groupby(["depth_group", "neuron_id"], as_index=False)
    .agg(
        n_sessions=("session_label", "nunique"),
        n_spikes=("n_isolated", "sum"),
    )
)

selected_neurons = {}

for grp in DEPTH_ORDER:
    manual = EXAMPLE_NEURONS.get(grp)

    if manual is not None:
        if manual not in set(wave_cells["neuron_id"]):
            raise ValueError(
                f"{manual!r} not found among eligible longitudinal neurons."
            )
        selected_neurons[grp] = manual
        continue

    candidates = (
        coverage[coverage["depth_group"] == grp]
        .sort_values(["n_sessions", "n_spikes"], ascending=False)
    )

    selected_neurons[grp] = (
        None
        if candidates.empty
        else candidates.iloc[0]["neuron_id"]
    )

display(pd.DataFrame({
    "depth_group": DEPTH_ORDER,
    "depth_label": [DEPTH_LABELS[g] for g in DEPTH_ORDER],
    "neuron_id": [selected_neurons[g] for g in DEPTH_ORDER],
    "sessions": [
        0 if selected_neurons[g] is None else
        wave_cells.loc[
            wave_cells["neuron_id"] == selected_neurons[g],
            "session_label",
        ].nunique()
        for g in DEPTH_ORDER
    ],
}))

# ---------- Extract spike waveforms ----------
# Store:
#   tw
#   median waveform
#   up to MAX_PLOTTED_SPIKES individual waveforms
#   total number of eligible spikes
example_waveforms = {}

for grp, neuron_id in selected_neurons.items():
    if neuron_id is None:
        continue

    neuron_cells = wave_cells[
        wave_cells["neuron_id"] == neuron_id
    ]

    for session_label in SESSION_GRID:
        rows = neuron_cells[
            neuron_cells["session_label"].astype(str) == session_label
        ]

        if rows.empty:
            continue

        cell = rows.iloc[0]
        spike_times = spike_groups[
            (cell.session_id, cell.dmd, cell.roi)
        ]

        with h5py.File(session_to_h5[cell.session_id], "r") as h5:
            g = h5[f"DMD{int(cell.dmd)}"]
            t = np.asarray(g["timebase_sec"][:], float)
            dff = g["dff"]

            trace = np.asarray(
                dff[int(cell.roi)]
                if dff.shape[-1] == len(t)
                else dff[:, int(cell.roi)],
                float,
            )

        fs = 1 / np.median(np.diff(t))

        search = int(round(
            ALIGN_WINDOW_MS / 1000 * fs
        ))
        pre = int(round(
            abs(WAVEFORM_WINDOW_MS[0]) / 1000 * fs
        ))
        post = int(round(
            WAVEFORM_WINDOW_MS[1] / 1000 * fs
        ))

        idx = np.searchsorted(t, spike_times)
        idx = idx[
            (idx >= search)
            & (idx < len(trace) - search)
        ]

        aligned_idx = np.array([
            i - search
            + np.argmax(trace[i-search:i+search+1])
            for i in idx
            if pre <= i < len(trace) - post
        ], dtype=int)

        waves = np.array([
            trace[i-pre:i+post]
            - np.median(trace[i-pre:i])
            for i in aligned_idx
            if pre <= i < len(trace) - post
        ])

        if len(waves) < MIN_SPIKES:
            continue

        waveform = np.median(waves, axis=0)
        tw = np.arange(-pre, post) / fs * 1000

        n_plot = min(MAX_PLOTTED_SPIKES, len(waves))

        if len(waves) > n_plot:
            keep = RNG.choice(
                len(waves),
                size=n_plot,
                replace=False,
            )
            plot_waves = waves[keep]
        else:
            plot_waves = waves

        example_waveforms[(grp, session_label)] = {
            "time": tw,
            "median": waveform,
            "spikes": plot_waves,
            "n_total": len(waves),
        }

if not example_waveforms:
    raise ValueError(
        "No eligible longitudinal waveform examples were found."
    )

# ---------- Common axes limits ----------
# Deliberately use the MEDIAN waveforms rather than noisy individual spikes.
all_medians = np.concatenate([
    item["median"]
    for item in example_waveforms.values()
])

ymin, ymax = np.nanmin(all_medians), np.nanmax(all_medians)
ypad = 0.08 * max(ymax - ymin, 1e-6)
ylims = (ymin - ypad, ymax + ypad)

In [ ]:
# ---------- Plot ----------
fig, axes = plt.subplots(
    3, 6,
    figsize=(7.5, 6),
    sharex=True,
    sharey=True,
)

for r, grp in enumerate(DEPTH_ORDER):
    color = DEPTH_COLORS[grp]

    for c, session_label in enumerate(SESSION_GRID):
        ax = axes[r, c]
        ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
        ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
        sns.despine(ax=ax)

        ax.axvline(
            0,
            color="0.7",
            ls="--",
            lw=0.8,
            zorder=0,
        )

        item = example_waveforms.get(
            (grp, session_label)
        )

        if item is not None:
            tw = item["time"]
            waveform = item["median"]
            waves = item["spikes"]

            # Individual isolated spikes.
            # Transpose so matplotlib draws one column = one waveform.
            ax.plot(
                tw,
                waves.T,
                color=color,
                lw=SPIKE_LINEWIDTH,
                alpha=SPIKE_ALPHA,
                zorder=1,
            )

            # Median waveform.
            ax.plot(
                tw,
                waveform,
                color='k',
                lw=2.3,
                zorder=3,
            )

#             ax.text(
#                 0.96,
#                 0.94,
#                 f'n={item["n_total"]:,}',
#                 transform=ax.transAxes,
#                 ha="right",
#                 va="top",
#                 fontsize=6.5,
#                 color="0.35",
#             )

        else:
            ax.text(
                0.5,
                0.5,
                "—",
                transform=ax.transAxes,
                ha="center",
                va="center",
                color="0.65",
                fontsize=14,
            )

        ax.set_xlim(WAVEFORM_WINDOW_MS)
        ax.set_ylim(ylims)
        ax.tick_params(
            axis="both",
            labelsize=8,
            length=2,
        )

        if r == 0:
            ax.set_title(
                session_label,
                fontsize=11,
            )

        if c == 0:
            ax.set_ylabel(
                DEPTH_LABELS[grp],
                color=color,
                fontsize=10,
            )

        ax.label_outer()

fig.supxlabel(
    "Time from spike peak (ms)",
    fontsize=15,y=0.05
)
fig.supylabel(
    "ΔF/F$_0$",
    fontsize=15,x=0.02
)

fig.suptitle(
    "Example spike waveforms by session",
    fontsize=18,y=1.0
)

fig.tight_layout()

filen = "Longitudinal_example_spike_waveforms_with_spikes"
save_figure(
    fig,
    os.path.join(SAVE_PATH, filen),
    formats=[".pdf"],
    dpi=300,
)

## Spike synchrony

STTC quantifies near-coincident spike timing while accounting for firing rate; complementary plots examine its temporal scale, depth structure, and session dependence.


In [ ]:
# Pairwise STTC heatmap — single session

# ---------- Parameters ----------

subject_id = "852835"
session_label = "A0"

METRIC = "spike_sttc_10ms"

DEPTH_COLORS = {
    "<100 µm": "#EBA287",
    "100–150 µm": "#d1e2b0",
    ">150 µm": "#7bbcd5",
}


# ---------- Resolve session ----------

SESSION_ID = sessions.loc[
    (sessions["subject_id"].astype(str) == str(subject_id)) &
    (sessions["session_label"].astype(str) == str(session_label)),
    "session_id",
].iloc[0]


# ---------- Pairwise data ----------

p = synchrony_pairs[
    synchrony_pairs["session_id"].astype(str) == str(SESSION_ID)
].copy()

if p.empty:
    raise ValueError(f"No synchrony pairs found for {SESSION_ID}")


# ---------- Build ROI metadata ----------

cells_a = (
    p[["dmd_a", "roi_a", "depth_a_um"]]
    .rename(columns={
        "dmd_a": "dmd",
        "roi_a": "roi",
        "depth_a_um": "depth_um",
    })
)

cells_b = (
    p[["dmd_b", "roi_b", "depth_b_um"]]
    .rename(columns={
        "dmd_b": "dmd",
        "roi_b": "roi",
        "depth_b_um": "depth_um",
    })
)

cells = (
    pd.concat([cells_a, cells_b], ignore_index=True)
    .drop_duplicates(subset=["dmd", "roi"])
    .sort_values(["depth_um", "dmd", "roi"])
    .reset_index(drop=True)
)

cells["key"] = list(zip(cells["dmd"], cells["roi"]))
cells["label"] = [
    f"D{int(dmd)}:{int(roi)}"
    for dmd, roi in cells["key"]
]

cells["depth_group"] = np.select(
    [
        cells["depth_um"] < 100,
        cells["depth_um"] > 150,
    ],
    [
        "<100 µm",
        ">150 µm",
    ],
    default="100–150 µm",
)


# ---------- Build symmetric matrix ----------

n = len(cells)

matrix = np.full((n, n), np.nan)

index = {
    key: i
    for i, key in enumerate(cells["key"])
}

for row in p.itertuples(index=False):

    a = (row.dmd_a, row.roi_a)
    b = (row.dmd_b, row.roi_b)

    if a not in index or b not in index:
        continue

    i = index[a]
    j = index[b]

    value = getattr(row, METRIC)

    matrix[i, j] = value
    matrix[j, i] = value


# ---------- Plot ----------

fig, ax = plt.subplots(figsize=(4.2, 3))
ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=8)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=8)
sns.heatmap(
    matrix,
    cmap="RdBu_r",
    center=0,
    vmin=-0.2,
    vmax=0.2,
    square=True,
    xticklabels=cells["label"],
    yticklabels=cells["label"],
    linewidths=0,
    cbar_kws={
        "label": "STTC (±50 ms)",
#         "label": 'Spike time correlation',
        "shrink": 0.8,
    },
    ax=ax,
)
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=8)
cbar.ax.set_ylabel("STTC (±50 ms)", fontsize=12)
ax.set_xlabel("")
ax.set_ylabel("")

# Color ROI labels by depth group
for tick, group in zip(ax.get_xticklabels(), cells["depth_group"]):
    tick.set_color(DEPTH_COLORS[group])

for tick, group in zip(ax.get_yticklabels(), cells["depth_group"]):
    tick.set_color(DEPTH_COLORS[group])


# ---------- Mark depth-group boundaries ----------

depth_groups = cells["depth_group"].to_numpy()

for i in range(1, len(depth_groups)):
    if depth_groups[i] != depth_groups[i - 1]:
        ax.axhline(i, color="k", lw=1.2)
        ax.axvline(i, color="k", lw=1.2)


ax.set_title(
    'Example session STTC',fontsize=15
)

fig.tight_layout()

filen = f"{subject_id}_{session_label}_STTC_heatmap"

save_figure(
    fig,
    os.path.join(SAVE_PATH, filen),
    formats=[".pdf"],
    dpi=300,
)

In [ ]:
# Within-depth-group STTC across mice for one session

# ---------- Parameters ----------

SESSION_LABEL = "A1"
METRIC = "spike_sttc_10ms"

GROUP_ORDER = ["<100 µm", "100–150 µm", ">150 µm"]

DEPTH_COLORS = {
    "<100 µm": "#EBA287",
    "100–150 µm": "#d1e2b0",
    ">150 µm": "#7bbcd5",
}


# ---------- Data ----------

plot_df = synchrony_pairs[
    synchrony_pairs["session_label"].astype(str) == SESSION_LABEL
].copy()

plot_df = plot_df.dropna(
    subset=[METRIC, "depth_a_um", "depth_b_um"]
)


# Assign depth group independently to each member of pair

def depth_group(depth):
    if depth < 100:
        return "<100 µm"
    elif depth > 150:
        return ">150 µm"
    return "100–150 µm"


plot_df["depth_group_a"] = plot_df["depth_a_um"].apply(depth_group)
plot_df["depth_group_b"] = plot_df["depth_b_um"].apply(depth_group)


# ---------- Keep only within-depth-group pairs ----------

plot_df = plot_df[
    plot_df["depth_group_a"] == plot_df["depth_group_b"]
].copy()

plot_df["depth_group"] = plot_df["depth_group_a"]


# Optional: remove pairs representing the same registered neuron
if "same_global_cell" in plot_df.columns:
    plot_df = plot_df[~plot_df["same_global_cell"]].copy()


# ---------- Mouse-level summaries ----------

mouse_medians = (
    plot_df
    .groupby(
        ["subject_id", "depth_group"],
        observed=True,
        as_index=False,
    )[METRIC]
    .median()
)


# ---------- Plot ----------

fig, ax = plt.subplots(figsize=(3.5, 3.5))
sns.despine()


# Violin distribution of all pairwise scores

sns.violinplot(
    data=plot_df,
    x="depth_group",
    y=METRIC,
    order=GROUP_ORDER,
    palette=DEPTH_COLORS,
#     inner=None,
#     cut=0,
    width=0.4,
    linewidth=1.5,
    ax=ax,
)


# Transparent fill, black edge

import matplotlib.colors as mcolors

for violin, color in zip(
    ax.collections[:3],
    [DEPTH_COLORS[g] for g in GROUP_ORDER],
):
    violin.set_facecolor(
        mcolors.to_rgba(color, 0.25)
    )
    violin.set_edgecolor("black")
    violin.set_linewidth(1.5)


# All individual ROI-pair observations

sns.stripplot(
    data=plot_df,
    x="depth_group",
    y=METRIC,
    order=GROUP_ORDER,
    palette=DEPTH_COLORS,
    size=7,
    alpha=1.0,
    jitter=0.18,
    ax=ax,
)


# Mouse medians

# sns.stripplot(
#     data=mouse_medians,
#     x="depth_group",
#     y=METRIC,
#     order=GROUP_ORDER,
#     color="black",
#     marker="D",
#     size=6,
#     jitter=0.10,
#     ax=ax,
# )


# ---------- Aesthetics ----------

ax.axhline(
    0,
    color="0.6",
    lw=1,
    linestyle="--",
    zorder=0,
)

ax.set_xlabel("Depth bin", fontsize=14)
ax.set_ylabel("Spike time corr.", fontsize=14)

ax.tick_params(
    axis="both",
    labelsize=10,
)

for spine in ax.spines.values():
    spine.set_linewidth(2)

fig.tight_layout()

filen = f"{SESSION_LABEL}_withinDepth_STTC"

save_figure(
    fig,
    os.path.join(SAVE_PATH, filen),
    formats=[".pdf"],
    dpi=300,
)

### STTC timescale

Sweep the STTC coincidence window to identify the timescale over which coordinated firing emerges.


In [ ]:
# STTC timescale sweep
# Across mice for one session label; only within-depth-group pairs

import h5py

from vip_slap2_analysis.voltage.analysis import (
    sttc,
    _observed_intervals,
    _intersect_intervals,
)

# ---------- Parameters ----------

SESSION_LABEL = "A0"
SUBJECT_ID = None       # None = all mice with this session
DT_WINDOWS_MS = np.array([2, 5, 10, 20, 40, 80, 160])

GROUP_ORDER = ["<100 µm", "100–150 µm", ">150 µm"]

DEPTH_COLORS = {
    "<100 µm": "#EBA287",
    "100–150 µm": "#d1e2b0",
    ">150 µm": "#7bbcd5",
}


def depth_group(depth):
    if depth < 100:
        return "<100 µm"
    elif depth > 150:
        return ">150 µm"
    return "100–150 µm"


# ---------- Select pairs ----------

pairs = synchrony_pairs[
    synchrony_pairs["session_label"].astype(str) == SESSION_LABEL
].copy()

if SUBJECT_ID is not None:
    pairs = pairs[
        pairs["subject_id"].astype(str) == str(SUBJECT_ID)
    ].copy()

pairs = pairs.dropna(
    subset=["depth_a_um", "depth_b_um"]
)

pairs["depth_group_a"] = pairs["depth_a_um"].apply(depth_group)
pairs["depth_group_b"] = pairs["depth_b_um"].apply(depth_group)

# within-depth-group pairs only
pairs = pairs[
    pairs["depth_group_a"] == pairs["depth_group_b"]
].copy()

pairs["depth_group"] = pairs["depth_group_a"]

# exclude same registered neuron if relevant
if "same_global_cell" in pairs.columns:
    pairs = pairs[~pairs["same_global_cell"]].copy()


# ---------- Spike-train lookup ----------

spike_lookup = {
    (str(session_id), int(dmd), int(roi)):
        g["spike_time_sec"].to_numpy(float)
    for (session_id, dmd, roi), g in
    spike_features.groupby(["session_id", "dmd", "roi"])
}


# ---------- Observation intervals ----------

trace_lookup = {
    str(row.session_id): row.trace_h5
    for row in sessions.itertuples(index=False)
}

interval_lookup = {}

for session_id, session_pairs in pairs.groupby("session_id"):

    sid = str(session_id)

    dmds = sorted(
        set(session_pairs["dmd_a"].astype(int))
        | set(session_pairs["dmd_b"].astype(int))
    )

    with h5py.File(trace_lookup[sid], "r") as h5:

        for dmd in dmds:

            g = h5[f"DMD{dmd}"]

            timebase = np.asarray(g["timebase_sec"][:], float)

            sample_epoch = (
                g["sample_epoch"][:]
                if "sample_epoch" in g
                else None
            )

            interval_lookup[(sid, dmd)] = _observed_intervals(
                timebase,
                sample_epoch,
            )


# ---------- STTC sweep ----------

rows = []

for pair in pairs.itertuples(index=False):

    sid = str(pair.session_id)

    train_a = spike_lookup.get(
        (sid, int(pair.dmd_a), int(pair.roi_a)),
        np.array([], float),
    )

    train_b = spike_lookup.get(
        (sid, int(pair.dmd_b), int(pair.roi_b)),
        np.array([], float),
    )

    intervals = _intersect_intervals(
        interval_lookup[(sid, int(pair.dmd_a))],
        interval_lookup[(sid, int(pair.dmd_b))],
    )

    for dt_ms in DT_WINDOWS_MS:

        rows.append({
            "subject_id": pair.subject_id,
            "session_id": pair.session_id,
            "session_label": pair.session_label,
            "pair_id": pair.pair_id,
            "dmd_a": pair.dmd_a,
            "roi_a": pair.roi_a,
            "dmd_b": pair.dmd_b,
            "roi_b": pair.roi_b,
            "depth_group": pair.depth_group,
            "dt_ms": dt_ms,
            "sttc": sttc(
                train_a,
                train_b,
                dt_ms / 1000,
                intervals,
            ),
        })


sttc_sweep = pd.DataFrame(rows)

display(
    sttc_sweep.groupby(
        ["depth_group", "dt_ms"]
    )["sttc"].agg(["count", "median"])
)

In [ ]:
# Plot STTC as a function of temporal window
# Thin lines = individual neuron-pair trajectories
# Heavy line = median across neuron pairs within each depth group

fig, ax = plt.subplots(figsize=(4.2, 3.5))

ax.tick_params(axis="x", which="major", reset=True, top=False, labelsize=12)
ax.tick_params(axis="y", which="major", reset=True, right=False, labelsize=12)
sns.despine()

# ---------- Individual neuron-pair trajectories ----------
# Include subject/session in grouping so pair IDs cannot collide across recordings.
pair_keys = [
    c for c in ["subject_id", "session_id", "pair_id", "depth_group"]
    if c in sttc_sweep.columns
]

for _, pair in sttc_sweep.groupby(pair_keys, observed=True):
    pair = pair.sort_values("dt_ms")
    group = pair["depth_group"].iloc[0]

    ax.plot(
        pair["dt_ms"],
        pair["sttc"],
        color=DEPTH_COLORS[group],
        alpha=0.12,
        lw=0.8,
        zorder=1,
    )

# ---------- Depth-group summary ----------
# Summarize directly over individual neuron pairs.
pair_summary = (
    sttc_sweep
    .groupby(["depth_group", "dt_ms"], observed=True)["sttc"]
    .median()
    .reset_index()
)

for group in GROUP_ORDER:
    g = pair_summary[pair_summary["depth_group"] == group]
    if g.empty:
        continue

    n_pairs = (
        sttc_sweep.loc[
            sttc_sweep["depth_group"] == group,
            pair_keys[:-1],   # identifiers excluding depth_group
        ]
        .drop_duplicates()
        .shape[0]
    )

    ax.plot(
        g["dt_ms"],
        g["sttc"],
        color=DEPTH_COLORS[group],
        marker="o",
        markersize=5,
        markeredgecolor="black",
        markeredgewidth=0.6,
        lw=2.5,
        label=f"{group}  (n={n_pairs} pairs)",
        zorder=3,
    )

ax.axhline(
    0,
    color="0.5",
    lw=1,
    linestyle="--",
    zorder=0,
)

ax.axvline(
    50,
    color="0.5",
    lw=1,
    linestyle=":",
    zorder=0,
)

ax.set_xscale("log")
ax.set_xticks(DT_WINDOWS_MS)
ax.set_xticklabels(DT_WINDOWS_MS)

ax.set_xlabel(
    "STTC coincidence window ±Δt (ms)",
    fontsize=13,
)
ax.set_ylabel(
    "STTC",
    fontsize=14,
)

ax.tick_params(
    axis="both",
    labelsize=10,
)

for spine in ax.spines.values():
    spine.set_linewidth(2)

ax.legend(
    frameon=False,
    fontsize=8,
)

ax.set_title(
    "Testing Δt across depth groups"
)

fig.tight_layout()

filen = f"{SESSION_LABEL}_STTC_timescale"
save_figure(
    fig,
    os.path.join(SAVE_PATH, filen),
    formats=[".pdf",".png"],
    dpi=300,
)

### Synchrony across sessions

Pool pairwise STTC within each depth relationship and summarize session-wise means with SEM. Pairwise observations are descriptive and are not treated as independent inferential replicates.


In [ ]:
# ============================================================
# Prepare session-wise STTC data
# ============================================================

METRIC = "spike_sttc_10ms"
EXCLUDE_SESSIONS = ["G1"]

GROUP_ORDER = ["<100 µm", "100–150 µm", ">150 µm"]

DEPTH_COLORS = {
    "<100 µm": "#EBA287",
    "100–150 µm": "#d1e2b0",
    ">150 µm": "#7bbcd5",
}


def depth_group(depth):
    if depth < 100:
        return "<100 µm"
    elif depth > 150:
        return ">150 µm"
    return "100–150 µm"


sttc_df = synchrony_pairs.dropna(
    subset=[METRIC, "depth_a_um", "depth_b_um"]
).copy()

sttc_df = sttc_df[
    ~sttc_df["session_label"].astype(str).isin(EXCLUDE_SESSIONS)
].copy()

sttc_df["depth_group_a"] = sttc_df["depth_a_um"].apply(depth_group)
sttc_df["depth_group_b"] = sttc_df["depth_b_um"].apply(depth_group)

if "same_global_cell" in sttc_df.columns:
    sttc_df = sttc_df[~sttc_df["same_global_cell"]].copy()


# ---------- Session ordering ----------

session_order = (
    sessions.loc[
        ~sessions["session_label"].astype(str).isin(EXCLUDE_SESSIONS),
        ["session_label", "session_order"]
    ]
    .drop_duplicates()
    .groupby("session_label", as_index=False)["session_order"]
    .median()
    .sort_values("session_order")
)

SESSION_ORDER = [
    s for s in session_order["session_label"]
    if s in sttc_df["session_label"].astype(str).unique()
]

sttc_df["session_label"] = pd.Categorical(
    sttc_df["session_label"],
    categories=SESSION_ORDER,
    ordered=True,
)

x = np.arange(len(SESSION_ORDER))

In [ ]:
# ============================================================
# Within-depth-group STTC across sessions
# Pooled pairwise mean ± SEM; subject_id intentionally ignored
# ============================================================

within_df = sttc_df[
    sttc_df["depth_group_a"] == sttc_df["depth_group_b"]
].copy()
within_df["depth_group"] = within_df["depth_group_a"]

summary_within = (
    within_df
    .groupby(["depth_group", "session_label"], observed=True)[METRIC]
    .agg(mean="mean", sem="sem", n_pairs="count")
    .reset_index()
)

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.tick_params(axis="x", which="major", reset=True, top=False, labelsize=12)
ax.tick_params(axis="y", which="major", reset=True, right=False, labelsize=12)
sns.despine()

for group in GROUP_ORDER:
    color = DEPTH_COLORS[group]
    g = (
        summary_within[summary_within["depth_group"] == group]
        .set_index("session_label")
        .reindex(SESSION_ORDER)
    )

    valid = g["mean"].notna().to_numpy()
    if not np.any(valid):
        continue

    xx = x[valid]
    mean = g["mean"].to_numpy()[valid]
    sem = g["sem"].to_numpy()[valid]

    ax.fill_between(
        xx, mean - sem, mean + sem,
        color=color, alpha=0.18, linewidth=0, zorder=1,
    )
    ax.plot(
        xx, mean, color=color, lw=3, marker="o", markersize=7,
        markeredgecolor="black", markeredgewidth=0.8,
        label=group, zorder=3,
    )

ax.set_xticks(x)
ax.set_xticklabels(SESSION_ORDER)
ax.set_xlabel("Session", fontsize=14)
ax.set_ylabel("Within-group STTC (±10 ms)", fontsize=14)
ax.tick_params(axis="both", labelsize=11)

for spine in ax.spines.values():
    spine.set_linewidth(2)

ax.legend(frameon=False, fontsize=10, title="Depth")
fig.tight_layout()

filen = "WithinDepth_STTC_acrossSessions"
save_figure(fig, os.path.join(SAVE_PATH, filen), formats=[".pdf"], dpi=300)

display(
    summary_within.pivot(
        index="session_label", columns="depth_group", values="n_pairs"
    )
)


In [ ]:
# ============================================================
# Between-depth-group STTC across sessions
# Pooled pairwise mean ± SEM; subject_id intentionally ignored
# ============================================================

import matplotlib.colors as mcolors

BETWEEN_PAIRS = [
    ("<100 µm", "100–150 µm"),
    ("<100 µm", ">150 µm"),
    ("100–150 µm", ">150 µm"),
]
LINESTYLES = ["-", "--", ":"]

def canonical_pair(a, b):
    for g1, g2 in BETWEEN_PAIRS:
        if {a, b} == {g1, g2}:
            return f"{g1}|{g2}"
    return np.nan

def blend_depth_colors(g1, g2):
    return tuple(
        (np.asarray(mcolors.to_rgb(DEPTH_COLORS[g1])) +
         np.asarray(mcolors.to_rgb(DEPTH_COLORS[g2]))) / 2
    )

between_df = sttc_df[
    sttc_df["depth_group_a"] != sttc_df["depth_group_b"]
].copy()
between_df["pair_type"] = [
    canonical_pair(a, b)
    for a, b in zip(between_df["depth_group_a"], between_df["depth_group_b"])
]
between_df = between_df.dropna(subset=["pair_type"])

summary_between = (
    between_df
    .groupby(["pair_type", "session_label"], observed=True)[METRIC]
    .agg(mean="mean", sem="sem", n_pairs="count")
    .reset_index()
)

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.tick_params(axis="x", which="major", reset=True, top=False, labelsize=12)
ax.tick_params(axis="y", which="major", reset=True, right=False, labelsize=12)
sns.despine()

for (g1, g2), ls in zip(BETWEEN_PAIRS, LINESTYLES):
    pair_type = f"{g1}|{g2}"
    g = (
        summary_between[summary_between["pair_type"] == pair_type]
        .set_index("session_label")
        .reindex(SESSION_ORDER)
    )

    valid = g["mean"].notna().to_numpy()
    if not np.any(valid):
        continue

    xx = x[valid]
    mean = g["mean"].to_numpy()[valid]
    sem = g["sem"].to_numpy()[valid]
    pair_color = blend_depth_colors(g1, g2)

    ax.fill_between(
        xx, mean - sem, mean + sem,
        color=pair_color, alpha=0.16, linewidth=0, zorder=1,
    )
    ax.plot(
        xx, mean, color=pair_color, linestyle=ls, lw=2.8, zorder=2,
    )
    ax.plot(
        xx, mean, linestyle="none", marker="o", markersize=10,
        markerfacecolor=DEPTH_COLORS[g1],
        markerfacecoloralt=DEPTH_COLORS[g2],
        fillstyle="left", markeredgecolor="black", markeredgewidth=0.8,
        label=f"{g1} ↔ {g2}", zorder=4,
    )

ax.set_xticks(x)
ax.set_xticklabels(SESSION_ORDER)
ax.set_xlabel("Session", fontsize=14)
ax.set_ylabel("Between-group STTC (±10 ms)", fontsize=14)
ax.tick_params(axis="both", labelsize=11)

for spine in ax.spines.values():
    spine.set_linewidth(2)

ax.legend(frameon=False, fontsize=9)
fig.tight_layout()

filen = "BetweenDepth_STTC_acrossSessions"
save_figure(fig, os.path.join(SAVE_PATH, filen), formats=[".pdf"], dpi=300)

display(
    summary_between.pivot(
        index="session_label", columns="pair_type", values="n_pairs"
    )
)
